In [10]:
# =====================================================================
# ChEMBL M3 activity curation + consensus labeling (Ki/IC50 actives)
# + more negatives (EC50 allowed for inactivity evidence)
# + export includes SMILES, Molecular Weight, (Molecular Formula if present)
#
# Input:  C:\Users\jdrew\Desktop\ChEMBL_M3_raw.csv  (semicolon-separated)
# Output: C:\Users\jdrew\Desktop\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv
# =====================================================================

import numpy as np
import pandas as pd
import csv

# ---------------------------------------------------------------------
# 1) LOAD FILE
# ---------------------------------------------------------------------
path = r"C:\Users\jdrew\Desktop\M3\ChEMBL_M3_raw.csv"

df = pd.read_csv(
    path,
    sep=";",
    engine="python",
    header=0,
    quoting=csv.QUOTE_NONE,
    escapechar="\\",
    on_bad_lines="skip",
)

print("Loaded df shape:", df.shape)
print("Columns (first 20):", df.columns.tolist()[:20])

# ---------------------------------------------------------------------
# 2) RENAME COLUMNS (activity + metadata)
# ---------------------------------------------------------------------
colmap = {
    # activity essentials
    "Molecule ChEMBL ID": "molecule_chembl_id",
    "Standard Type": "standard_type",
    "Standard Relation": "standard_relation",
    "Standard Value": "standard_value",
    "Standard Units": "standard_units",

    # metadata you want in final output
    "Smiles": "smiles",
    "Molecular Weight": "molecular_weight",
    "Molecular Formula": "molecular_formula",  # may not exist in this export
    "Molecule Name": "molecule_name",
}
df = df.rename(columns={k: v for k, v in colmap.items() if k in df.columns})

required = ["molecule_chembl_id", "standard_type", "standard_value", "standard_units", "standard_relation"]
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing required columns after rename: {missing}"

# ---------------------------------------------------------------------
# 3) HELPERS
# ---------------------------------------------------------------------
def _to_p_from_value(value, unit):
    """
    Convert value+unit to p-scale:
      p = -log10(M)
    Handles ChEMBL artifact "-M" as molar ("M").
    """
    if pd.isna(value):
        return np.nan
    try:
        v = float(value)
    except Exception:
        return np.nan
    if v <= 0:
        return np.nan

    u = str(unit).strip().lower()
    u = u.replace("μ", "u").replace("µ", "u")
    if u == "-m":  # artifact seen in some exports
        u = "m"

    factors = {
        "pm": 1e-12,
        "nm": 1e-9,
        "um": 1e-6,
        "mm": 1e-3,
        "m":  1.0,   # molar (M)
    }
    if u not in factors:
        return np.nan

    return -np.log10(v * factors[u])


def _relation_evidence(rel, p, p_active=6.0, p_inactive=5.0):
    """
    Translate relation + p into qualitative evidence.

    '< x'  -> true value could be more potent -> supports activity
    '> x'  -> true value could be less potent -> supports inactivity
    '='    -> treat by thresholds
    """
    if pd.isna(p):
        return "missing"

    r = "" if pd.isna(rel) else str(rel).strip()

    if r in ("<", "<="):
        return "active_strong" if p >= p_active else "active_weak"
    if r in (">", ">="):
        return "inactive_strong" if p <= p_inactive else "inactive_weak"

    if p >= p_active:
        return "active"
    if p <= p_inactive:
        return "inactive"
    return "gray"


# ---------------------------------------------------------------------
# 4) CONSENSUS LABELING (more negatives)
#    - Actives: only Ki/IC50 can create active votes
#    - Inactives: Ki/IC50 + EC50 can create inactive votes
# ---------------------------------------------------------------------
def label_consensus(
    df: pd.DataFrame,
    id_col: str = "molecule_chembl_id",
    type_col: str = "standard_type",
    value_col: str = "standard_value",
    unit_col: str = "standard_units",
    relation_col: str = "standard_relation",

    # robust thresholds
    p_active: float = 6.0,              # <= 1 µM
    p_inactive: float = 5.0,            # >= 10 µM

    # single-point thresholds (edit to your preference)
    p_active_single: float = 6.5,       # ~316 nM (use 6.4 for 400 nM, 6.3 for 500 nM)
    p_inactive_single: float = 5.0,     # 10 µM (relaxed)

    # robustness
    min_n: int = 2,
    frac_required: float = 0.75,

    # type rules
    active_types: tuple[str, ...] = ("Ki", "IC50"),          # only these create actives
    inactive_types: tuple[str, ...] = ("Ki", "IC50", "EC50") # these can support inactives
) -> pd.DataFrame:

    d = df.copy()
    d[type_col] = d[type_col].astype(str).str.strip()

    # keep only types we consider at all
    keep_types = set(active_types) | set(inactive_types)
    d = d[d[type_col].isin(keep_types)].copy()

    # compute p-values
    d["p"] = [_to_p_from_value(v, u) for v, u in zip(d[value_col], d[unit_col])]

    # evidence categories
    d["evidence"] = [
        _relation_evidence(r, p, p_active=p_active, p_inactive=p_inactive)
        for r, p in zip(d[relation_col], d["p"])
    ]

    # flags by type
    d["is_active_type"] = d[type_col].isin(active_types)
    d["is_inactive_type"] = d[type_col].isin(inactive_types)

    # votes
    d["v_active"] = (d["is_active_type"] & d["evidence"].isin(["active", "active_strong"])).astype(float)
    d["v_inactive"] = (d["is_inactive_type"] & d["evidence"].isin(["inactive", "inactive_strong"])).astype(float)

    # aggregate per compound
    def median_kiki(p_series):
        mask = d.loc[p_series.index, "is_active_type"].values
        vals = p_series.values[mask]
        if vals.size == 0 or np.all(np.isnan(vals)):
            return np.nan
        return float(np.nanmedian(vals))

    agg = (
        d.groupby(id_col, dropna=False)
        .agg(
            n=("p", lambda x: int(np.sum(~pd.isna(x)))),
            p_median=("p", "median"),               # median across kept types (Ki/IC50/EC50)
            p_median_kiki=("p", median_kiki),       # median restricted to Ki/IC50 only
            frac_active=("v_active", "mean"),
            frac_inactive=("v_inactive", "mean"),
            n_strong_active=("evidence", lambda x: int((pd.Series(x) == "active_strong").sum())),
            n_strong_inactive=("evidence", lambda x: int((pd.Series(x) == "inactive_strong").sum())),
            any_active_vote=("v_active", lambda x: bool(np.nansum(x) > 0)),
        )
        .reset_index()
    )
    # ---- ROUND p-values to 2 decimals (presentation only) ----
    agg["p_median"] = agg["p_median"].round(2)
    agg["p_median_kiki"] = agg["p_median_kiki"].round(2)
    
    # decision rules
    def consensus(row):
        # single-point labels
        if row["n"] == 1:
            # single active: must have Ki/IC50 potency
            if (not pd.isna(row["p_median_kiki"])) and (row["p_median_kiki"] >= p_active_single):
                return "active_single"

            # single inactive: allow EC50 too, but block if any active signal exists
            if (not row["any_active_vote"]) and (not pd.isna(row["p_median"])) and (row["p_median"] <= p_inactive_single):
                return "inactive_single"

            return "insufficient"

        # robust requires at least min_n measurements
        if row["n"] < min_n:
            return "insufficient"

        # contradictory strong censoring
        if row["n_strong_active"] > 0 and row["n_strong_inactive"] > 0:
            return "ambiguous"

        # robust active: Ki/IC50 median + agreement
        if (not pd.isna(row["p_median_kiki"])) and (row["p_median_kiki"] >= p_active) and (row["frac_active"] >= frac_required):
            return "active"

        # robust inactive: allow inactive evidence, but block if any active vote exists
        if (not row["any_active_vote"]) and (not pd.isna(row["p_median"])) and (row["p_median"] <= p_inactive) and (row["frac_inactive"] >= frac_required):
            return "inactive"

        return "ambiguous"

    agg["consensus_label"] = agg.apply(consensus, axis=1)
    return agg.sort_values(id_col).reset_index(drop=True)


# ---------------------------------------------------------------------
# 5) RUN CONSENSUS
# ---------------------------------------------------------------------
labels = label_consensus(df)

print("\nConsensus table shape:", labels.shape)
print(labels["consensus_label"].value_counts(dropna=False))


# ---------------------------------------------------------------------
# 6) BUILD METADATA TABLE (ONLY COLUMNS THAT EXIST) + MERGE
# ---------------------------------------------------------------------
def first_nonnull(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else pd.NA

# Choose metadata columns safely (avoid KeyError)
meta_cols = ["molecule_chembl_id"]
for c in ["smiles", "molecular_weight", "molecular_formula", "molecule_name"]:
    if c in df.columns:
        meta_cols.append(c)

agg_dict = {}
if "smiles" in meta_cols:
    agg_dict["smiles"] = first_nonnull
if "molecular_formula" in meta_cols:
    agg_dict["molecular_formula"] = first_nonnull
if "molecule_name" in meta_cols:
    agg_dict["molecule_name"] = first_nonnull
if "molecular_weight" in meta_cols:
    agg_dict["molecular_weight"] = "median"

meta = (
    df[meta_cols]
    .groupby("molecule_chembl_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

labels = labels.merge(meta, on="molecule_chembl_id", how="left")

# nicer column order
front = ["molecule_chembl_id"]
for c in ["smiles", "molecular_formula", "molecular_weight", "molecule_name"]:
    if c in labels.columns:
        front.append(c)
rest = [c for c in labels.columns if c not in front]
labels = labels[front + rest]


# ---------------------------------------------------------------------
# 7) SAVE
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# 8) DROP insufficient + ambiguous labels
# ---------------------------------------------------------------------
labels = labels[~labels["consensus_label"].isin(["insufficient", "ambiguous"])]

print("\nFiltered label counts:")
print(labels["consensus_label"].value_counts())

out_path = r"C:\Users\jdrew\Desktop\M3\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv"
labels.to_csv(out_path, index=False)
out_path = r"C:\Users\jdrew\Desktop\M3\ChEMBL_M3_consensus_DB.xlsx"
labels.to_excel(out_path, index=False)
print("[SAVED]", out_path)


Loaded df shape: (8544, 48)
Columns (first 20): ['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key', 'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value', 'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment', 'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI']

Consensus table shape: (3770, 10)
consensus_label
active_single      1502
insufficient       1412
inactive_single     463
active              286
ambiguous            90
inactive             17
Name: count, dtype: int64

Filtered label counts:
consensus_label
active_single      1502
inactive_single     463
active              286
inactive             17
Name: count, dtype: int64
[SAVED] C:\Users\jdrew\Desktop\M3\ChEMBL_M3_consensus_DB.xlsx
